#### Load data and create descriptive statistics

In [1]:
import pandas as pd
import sqlite3
import numpy as np

In [2]:
# Define the file path for the SQLite database
db_path = "E:\Masters of Data Science Program - All Materials\Semester 2 Classes_1_13_2025\Applied Database Technologies\Final Project\clinical_trials.db"

<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:2: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Sal\AppData\Local\Temp\ipykernel_15952\2233300648.py:2: SyntaxWarning: invalid escape sequence '\M'
  db_path = "E:\Masters of Data Science Program - All Materials\Semester 2 Classes_1_13_2025\Applied Database Technologies\Final Project\clinical_trials.db"


In [3]:
# Establish a connection to the SQLite database
conn = sqlite3.connect(db_path)

In [4]:
# Write the query to pull in the data in format needed
query = """
select
DISTINCT(a.ndc_code) as ndc,
a.drug_name,
a.active_ingredients,
a.strength,
b.labeler_name,
a.action_type as status,
a.action_date as ct_end_date,
b.marketing_start_date as gtm_date
from 
fda_approved a,
drug_data b
where a.ndc_code = b.product_ndc
and a.drug_name !='HUMIRA'
and a.ndc_code !='65219-556';
"""

In [5]:
# Execute the query and load the data into a DataFrame
df = pd.read_sql_query(query, conn)

# Close the database connection
conn.close()

In [6]:
df['ct_end_date'] = pd.to_datetime(df['ct_end_date'], format='%m/%d/%Y', errors='coerce')
df['gtm_date'] = pd.to_datetime(df['gtm_date'], format='%m/%d/%Y', errors='coerce')

In [7]:
df['days_to_brand_launch'] = df['gtm_date'] - df['ct_end_date']

In [8]:
avg_time_to_launch_days = df['days_to_brand_launch'].median()
in_years = avg_time_to_launch_days.days / 365.25
avg_time_to_launch_yrs = round(in_years, 2)



In [9]:
df['Years to Brand Launch'] = df['days_to_brand_launch'].dt.days / 365.25

In [10]:
# Rename columns
df.rename(columns={
    'ndc': 'NDC',
    'drug_name': 'Brand Name',
    'active_ingredients': 'Molecular Name',
    'strength': 'Strength',
    'labeler_name': 'Manufacturer',
    'status': 'Status',
    'ct_end_date': 'Clinical Trial End Date',
    'gtm_date': 'Brand Launch Date',
    'days_to_brand_launch': 'Days to Brand Launch'
}, inplace=True)

In [13]:
df.to_pickle('clinical_trials.pkl')